In [1]:
import time
import tracemalloc
import numpy as np

print("✅ Day 4: Profiling & Optimization tools loaded.")

✅ Day 4: Profiling & Optimization tools loaded.


In [3]:
'''The "Before" Code (Buggy & Slow)
A naive training loop with memory leaks and slow list comprehensions.'''
# 🐌 BEFORE: Unoptimized Training Workflow

def slow_batch_generator(data, batch_size, cache=[]):
    # Bug 1: Mutable default argument 'cache' causes memory leak
    # Bug 2: Loads ALL batches into memory at once instead of streaming
    batches = []
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        cache.append(batch) 
        batches.append(batch)
    return batches 

def slow_training_loop(data, epochs=1):
    for epoch in range(epochs):
        batches = slow_batch_generator(data, batch_size=1024)
        for batch in batches:
            # Bug 3: Slow Python for-loop math instead of vectorization
            loss = sum([x**2 for x in batch]) / len(batch)
    return loss

In [4]:
'''The "After" Code (Optimized)
 Memory-aware batching using generators and vectorized math.'''

# 🚀 AFTER: Optimized Memory-Aware Workflow

def fast_batch_generator(data, batch_size):
    # Fix 1 & 2: Removed mutable cache, using 'yield' for O(1) memory
    np.random.shuffle(data) # Added data shuffling for proper training
    for i in range(0, len(data), batch_size):
        yield data[i:i+batch_size]

def fast_training_loop(data, epochs=1):
    for epoch in range(epochs):
        # Fix 3: NumPy vectorization for instant math
        for batch in fast_batch_generator(data, batch_size=1024):
            loss = np.mean(batch**2)
    return loss

In [5]:
'''Timing & Memory Comparison (The Proof)
Profile both functions to show performance gains'''

# Generate dummy dataset (1 million floats)
dummy_data = np.random.rand(1000000)

# Measure Before
tracemalloc.start()
start = time.perf_counter()
slow_training_loop(dummy_data)
slow_time = time.perf_counter() - start
_, slow_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

# Measure After
tracemalloc.start()
start = time.perf_counter()
fast_training_loop(dummy_data)
fast_time = time.perf_counter() - start
_, fast_peak = tracemalloc.get_traced_memory()
tracemalloc.stop()

print("--- 📊 Optimization Results ---")
print(f"🐌 Before: Time = {slow_time:.3f}s | Peak Memory = {slow_peak / 1e6:.2f} MB")
print(f"🚀 After:  Time = {fast_time:.3f}s | Peak Memory = {fast_peak / 1e6:.2f} MB")
print(f"⚡ Speedup: {slow_time/fast_time:.1f}x faster")
print(f"💾 Memory Saved: {slow_peak/fast_peak:.1f}x less RAM")

--- 📊 Optimization Results ---
🐌 Before: Time = 0.533s | Peak Memory = 0.16 MB
🚀 After:  Time = 0.013s | Peak Memory = 0.01 MB
⚡ Speedup: 39.9x faster
💾 Memory Saved: 11.8x less RAM


In [7]:
'''Deliverable Summary'''

print("""
--- 🐛 3 Bugs Identified & Fixed ---
1. Bug: Memory Leak via mutable default arg (`cache=[]`).
   Fix: Removed it entirely. State shouldn't be held in function defaults.
2. Bug: Out-of-Memory (OOM) risk from appending all batches to a list.
   Fix: Replaced with a Generator (`yield`) to stream batches one-by-one.
3. Bug: High CPU latency due to Python list comprehensions for math.
   Fix: Swapped to `np.mean()` for fast C-level vectorization.

--- 🛠️ Debugging & Optimization Checklist ---
1. Are data pipelines using generators instead of lists?
2. Have mutable default arguments been eliminated?
3. Is math vectorized using NumPy/PyTorch instead of pure Python loops?
4.Did I profile memory usage with `tracemalloc`?
5. Is the data being shuffled per epoch to prevent model overfitting?
""")


--- 🐛 3 Bugs Identified & Fixed ---
1. Bug: Memory Leak via mutable default arg (`cache=[]`).
   Fix: Removed it entirely. State shouldn't be held in function defaults.
2. Bug: Out-of-Memory (OOM) risk from appending all batches to a list.
   Fix: Replaced with a Generator (`yield`) to stream batches one-by-one.
3. Bug: High CPU latency due to Python list comprehensions for math.
   Fix: Swapped to `np.mean()` for fast C-level vectorization.

--- 🛠️ Debugging & Optimization Checklist ---
1. Are data pipelines using generators instead of lists?
2. Have mutable default arguments been eliminated?
3. Is math vectorized using NumPy/PyTorch instead of pure Python loops?
4.Did I profile memory usage with `tracemalloc`?
5. Is the data being shuffled per epoch to prevent model overfitting?

